# Листок № 4. Классическое, статистическое, геометрическое определение вероятности


## Задание 4.1. Выборка шаров из корзины

**Условие:**
Корзина содержит 23 шара: 8 белых, 6 синих и 9 красных. На каждом шаге из корзины «наудачу» извлекается шар и назад в корзину не возвращается. Исход $n$ последовательных извлечений называется выборкой объема $n$ без возвращения или бесповторной выборкой. Из корзины наудачу (без возвращения) извлечены 6 шаров. Найдите вероятности следующих событий и покажите их статистическую устойчивость:
- a) $A = \{\text{все шары красные}\}$;
- б) $B = \{\text{3 синих, 2 белых и 1 красный}\}$;
- в) $D = \{\text{в точности 4 белых шара}\}$.
- г) Разобрать случай извлечения шаров с возвращением. Найдите вероятность $P(A)$ события $A$, если «статистическая вероятность» равна $\hat{p}(A) \approx 0{,}00358549$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

total_balls = 23
w, b, r = 8, 6, 9
n_draw = 6

p_A_theor = comb(r, 6) / comb(total_balls, n_draw)
p_B_theor = (comb(b, 3) * comb(w, 2) * comb(r, 1)) / comb(total_balls, n_draw)
p_D_theor = (comb(w, 4) * comb(b + r, 2)) / comb(total_balls, n_draw)
p_A_repl_theor = (r / total_balls) ** n_draw

print(f"P(A) = {p_A_theor:.10f} ({comb(r, 6)}/{comb(total_balls, n_draw)})")
print(f"P(B) = {p_B_theor:.10f} ({comb(b, 3)*comb(w, 2)*comb(r, 1)}/{comb(total_balls, n_draw)})")
print(f"P(D) = {p_D_theor:.10f} ({comb(w, 4)*comb(b + r, 2)}/{comb(total_balls, n_draw)})")
print(f"P(A с возвращением) = {p_A_repl_theor:.10f}")

N = 150000
urn = np.array([0] * w + [1] * b + [2] * r)

perms = np.random.rand(N, total_balls).argsort(axis=1)[:, :n_draw]
draws = urn[perms]

cnt_w = np.sum(draws == 0, axis=1)
cnt_b = np.sum(draws == 1, axis=1)
cnt_r = np.sum(draws == 2, axis=1)

ev_A = (cnt_r == 6)
ev_B = (cnt_b == 3) & (cnt_w == 2) & (cnt_r == 1)
ev_D = (cnt_w == 4)

draws_repl = np.random.choice(urn, size=(N, n_draw), replace=True)
ev_A_repl = np.sum(draws_repl == 2, axis=1) == 6

freq_A = np.cumsum(ev_A) / np.arange(1, N + 1)
freq_A_repl = np.cumsum(ev_A_repl) / np.arange(1, N + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(freq_A, label="Частота (без возвращения)")
axes[0].axhline(p_A_theor, color="red", linestyle="--", label=f"Теория = {p_A_theor:.6f}")
axes[0].set_title("Событие A (без возвращения)")
axes[0].set_xlabel("N")
axes[0].set_ylabel("Относительная частота")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(freq_A_repl, label="Частота (с возвращением)", color="green")
axes[1].axhline(p_A_repl_theor, color="red", linestyle="--", label=f"Теория = {p_A_repl_theor:.6f}")
axes[1].set_title("Событие A (с возвращением)")
axes[1].set_xlabel("N")
axes[1].set_ylabel("Относительная частота")
axes[1].grid(True)
axes[1].legend()
plt.tight_layout()
plt.show()


## Задание 4.2. Рассаживание мужчин и женщин в ряд

**Условие:**
5 мужчин и 5 женщин случайным образом рассаживаются в ряд на 10 мест. Найдите вероятности следующих событий:
- $A$ – любые две женщины не сидят рядом;
- $B$ – все мужчины сидят рядом.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

p_A_theor = 6 / comb(10, 5)
p_B_theor = 6 / comb(10, 5)

print(f"P(A) теория = {p_A_theor:.6f} (1/42)")
print(f"P(B) теория = {p_B_theor:.6f} (1/42)")

N = 100000
row = np.array([0] * 5 + [1] * 5)
perms = np.random.rand(N, 10).argsort(axis=1)
seated = row[perms]

has_adj_women = np.any((seated[:, :-1] == 1) & (seated[:, 1:] == 1), axis=1)
ev_A = ~has_adj_women

ev_B = np.zeros(N, dtype=bool)
for start in range(6):
    ev_B |= np.all(seated[:, start:start+5] == 0, axis=1)

p_A_sim = np.mean(ev_A)
p_B_sim = np.mean(ev_B)

print(f"P(A) оценка = {p_A_sim:.6f}")
print(f"P(B) оценка = {p_B_sim:.6f}")

freq_A = np.cumsum(ev_A) / np.arange(1, N + 1)
freq_B = np.cumsum(ev_B) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(freq_A, label=f"Частота P(A) (оценка = {p_A_sim:.5f})", color="blue")
ax.plot(freq_B, label=f"Частота P(B) (оценка = {p_B_sim:.5f})", color="purple")
ax.axhline(p_A_theor, color="red", linestyle="--", label=f"Теория 1/42 ≈ {p_A_theor:.5f}")
ax.set_title("Сходимость частот событий A и B (рассаживание в ряд)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.3. Бросание симметричной монеты до первого герба

**Условие:**
Предположим, что «симметричную» монету бросают до первого появления герба. Найдите вероятность события $A$, состоящего в том, что будет произведено не более пяти бросаний.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_theor = 1 - (1 / 2) ** 5
print(f"Теория P(A) = {p_theor:.5f} (31/32)")

N = 100000
tosses = np.random.geometric(p=0.5, size=N)
ev_A = tosses <= 5
p_sim = np.mean(ev_A)
print(f"Оценка Монте-Карло P(A) = {p_sim:.5f}")

freq = np.cumsum(ev_A) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq, label=f"Частота (оценка = {p_sim:.5f})", color="navy")
ax.axhline(p_theor, color="red", linestyle="--", label=f"Теория = {p_theor:.5f}")
ax.set_title("Сходимость частоты события A (не более 5 бросаний)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.4. Бросание симметричной монеты до двух одинаковых сторон подряд

**Условие:**
Предположим, что «симметричную» монету бросают до тех пор, пока она не выпадет дважды одной и той же стороной подряд. Построить вероятностную модель т.е. $(\Omega, P(\cdot))$ и найдите вероятность того, что потребуется чётное число бросаний.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_theor = 2 / 3
print(f"Теоретическая вероятность = {p_theor:.6f} (2/3)")

N = 100000
steps = np.random.geometric(p=0.5, size=N)
total_tosses = steps + 1

ev_even = (total_tosses % 2 == 0)
p_sim = np.mean(ev_even)
print(f"Оценка Монте-Карло = {p_sim:.6f}")

freq = np.cumsum(ev_even) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq, label=f"Частота (оценка = {p_sim:.5f})", color="teal")
ax.axhline(p_theor, color="red", linestyle="--", label=f"Теория = 2/3 ≈ {p_theor:.5f}")
ax.set_title("Сходимость частоты четного числа бросаний (симметричная монета)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.5. Бросание несимметричной монеты до двух одинаковых сторон подряд

**Условие:**
Предположим, что «несимметричную» («неправильную») монету бросают до тех пор, пока она не выпадет дважды одной и той же стороной подряд (вероятность выпадения «герба» равна $p = 0{,}7$). Постройте вероятностную модель, т.е. $(\Omega, P(\cdot))$ и найдите вероятность того, что потребуется:
- а) чётное число бросаний;
- б) число бросаний кратно 4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = 0.7
q = 1 - p

p_even_theor = (p**2 + q**2) / (1 - p * q)
p_mult4_theor = (p * q * (p**2 + q**2)) / (1 - (p * q)**2)

print(f"Теория а) четное число = {p_even_theor:.6f} (58/79)")
print(f"Теория б) кратно 4 = {p_mult4_theor:.6f} (1218/9559)")

N = 100000
max_len = 40
tosses = (np.random.rand(N, max_len) < p).astype(int)
same_adj = (tosses[:, :-1] == tosses[:, 1:])
first_match_idx = np.argmax(same_adj, axis=1) + 2

ev_even = (first_match_idx % 2 == 0)
ev_mult4 = (first_match_idx % 4 == 0)

print(f"Оценка Монте-Карло а) = {np.mean(ev_even):.6f}")
print(f"Оценка Монте-Карло б) = {np.mean(ev_mult4):.6f}")

freq_even = np.cumsum(ev_even) / np.arange(1, N + 1)
freq_mult4 = np.cumsum(ev_mult4) / np.arange(1, N + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(freq_even, label=f"Оценка = {np.mean(ev_even):.5f}", color="royalblue")
axes[0].axhline(p_even_theor, color="red", linestyle="--", label=f"Теория = {p_even_theor:.5f}")
axes[0].set_title("Четное число бросаний")
axes[0].set_xlabel("N")
axes[0].set_ylabel("Относительная частота")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(freq_mult4, label=f"Оценка = {np.mean(ev_mult4):.5f}", color="darkorange")
axes[1].axhline(p_mult4_theor, color="red", linestyle="--", label=f"Теория = {p_mult4_theor:.5f}")
axes[1].set_title("Число бросаний кратно 4")
axes[1].set_xlabel("N")
axes[1].set_ylabel("Относительная частота")
axes[1].grid(True)
axes[1].legend()
plt.tight_layout()
plt.show()


## Задание 4.6. Задача о встрече двух лиц с разным временем ожидания

**Условие:**
Два лица $A$ и $B$ условились встретиться в определенном месте между двумя и тремя часами дня. Лицо $A$ ждет другого в течение 10 минут, после чего уходит; лицо $B$ ждет другого в течение 12 минут. Считая, что момент прихода на встречу выбирается каждым «наудачу» в пределах указанного часа, найдите вероятность, что встреча состоится. Постройте график зависимости относительной частоты $\hat{p}(A) = \frac{N(A)}{N}$ события $A$ от числа проведенных реализаций опыта $N = 150\,000$ и приведете оценку вероятности $\hat{p}(A)$ для $N = 150\,000$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_theor = (3600 - 0.5 * 50**2 - 0.5 * 48**2) / 3600
print(f"Теоретическая вероятность = {p_theor:.6f} (1198/3600 = 599/1800)")

N = 150000
x = np.random.uniform(0, 60, N)
y = np.random.uniform(0, 60, N)

meet = (y - x <= 10) & (x - y <= 12)
p_sim = np.mean(meet)
print(f"Оценка Монте-Карло при N={N}: p_hat = {p_sim:.6f}")

freq = np.cumsum(meet) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq, label=f"Относительная частота p_hat (оценка = {p_sim:.6f})", color="mediumblue")
ax.axhline(p_theor, color="red", linestyle="--", label=f"Теория = {p_theor:.6f}")
ax.set_title("Зависимость относительной частоты от числа опытов N = 150 000")
ax.set_xlabel("N")
ax.set_ylabel("p_hat")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.7. Задача о встрече Ивана и Петра

**Условие:**
Иван и Пётр договорились о встрече в определённом месте между 11 и 12 часами. Каждый приходит и ждёт другого до истечения часа, но не более 15 минут. Найдите вероятности следующих событий:
- $A$ – встреча состоялась;
- $B$ – встреча не состоялась;
- $C$ – Ивану не пришлось ждать Петра;
- $D$ – встреча состоялась после 11 ч. 30 мин;
- $E$ – Иван опоздал на встречу;
- $F$ – встреча состоялась, когда до истечения часа оставалось меньше 5 минут.

Постройте график зависимости относительной частоты $\hat{p}(A) = \frac{N(A)}{N}$ события $A$ от числа проведенных реализаций опыта $N = 150\,000$ и приведете оценку вероятности $\hat{p}(A)$ для $N = 150\,000$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_A_theor = (3600 - 45**2) / 3600
p_B_theor = 1 - p_A_theor
p_C_theor = 0.5 * p_A_theor

N = 150000
x = np.random.uniform(0, 60, N)
y = np.random.uniform(0, 60, N)

ev_A = np.abs(x - y) <= 15
ev_B = ~ev_A
ev_C = ev_A & (y <= x)
ev_D = ev_A & (np.maximum(x, y) >= 30)
ev_E = ev_A & (x > y)
ev_F = ev_A & (np.maximum(x, y) >= 55)

print(f"P(A) = {np.mean(ev_A):.5f} (теория: {p_A_theor:.5f})")
print(f"P(B) = {np.mean(ev_B):.5f} (теория: {p_B_theor:.5f})")
print(f"P(C) = {np.mean(ev_C):.5f} (теория: {p_C_theor:.5f})")
print(f"P(D) = {np.mean(ev_D):.5f}")
print(f"P(E) = {np.mean(ev_E):.5f}")
print(f"P(F) = {np.mean(ev_F):.5f}")

freq_A = np.cumsum(ev_A) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq_A, label=f"Частота P(A) (оценка = {np.mean(ev_A):.5f})", color="darkcyan")
ax.axhline(p_A_theor, color="red", linestyle="--", label=f"Теория = {p_A_theor:.5f}")
ax.set_title("Сходимость частоты события A (встреча состоялась, N = 150 000)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.8. Выбор двух зависимых точек на отрезке

**Условие:**
На отрезке $[a,b]$, лежащем на числовой оси, наудачу выбирается первая точка с координатой $x$, после чего на отрезке $[a,x]$ наудачу выбирается вторая точка с координатой $y$. Найдите вероятности следующих событий:
- $A$ – вторая точка ближе к $b$, чем первая к $a$;
- $B$ – расстояние между точками $x$ и $y$ меньше половины длины отрезка;
- $C$ – первая точка ближе ко второй, чем к точке $b$.

Постройте график зависимости относительной частоты $\hat{p}(A) = \frac{N(A)}{N}$ события $A$ от числа проведенных реализаций опыта $N = 150\,000$ и приведете оценку вероятности $\hat{p}(A)$ для $N = 150\,000$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_A_theor = 1 - np.log(2)
p_B_theor = 0.5 * (1 + np.log(2))
p_C_theor = np.log(2)

print(f"Теория P(A) = {p_A_theor:.6f} (1 - ln 2)")
print(f"Теория P(B) = {p_B_theor:.6f} ((1 + ln 2)/2)")
print(f"Теория P(C) = {p_C_theor:.6f} (ln 2)")

N = 150000
x = np.random.uniform(0, 1, N)
y = np.random.uniform(0, x)

ev_A = y > 1 - x
ev_B = (x - y) < 0.5
ev_C = (x - y) < (1 - x)

print(f"Оценка P(A) = {np.mean(ev_A):.6f}")
print(f"Оценка P(B) = {np.mean(ev_B):.6f}")
print(f"Оценка P(C) = {np.mean(ev_C):.6f}")

freq_A = np.cumsum(ev_A) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq_A, label=f"Частота P(A) (оценка = {np.mean(ev_A):.6f})", color="forestgreen")
ax.axhline(p_A_theor, color="red", linestyle="--", label=f"Теория = {p_A_theor:.6f}")
ax.set_title("Сходимость частоты события A (N = 150 000)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.9. Разбиение отрезка на три части и остроугольный треугольник

**Условие:**
Отрезок $[0; 1]$ случайным образом делится на 3 части. Найти вероятность того, что из полученных отрезков можно сложить треугольник и этот треугольник будет остроугольным. Постройте график зависимости относительной частоты $\hat{p}(A) = \frac{N(A)}{N}$ события $A$ от числа проведенных реализаций опыта $N = 150\,000$ и приведете оценку вероятности $\hat{p}(A)$ для $N = 150\,000$. Найдите аналитическое решение для вероятности.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_theor = 3 * np.log(2) - 2
print(f"Теоретическая вероятность = {p_theor:.10f} (3*ln(2) - 2)")

N = 150000
pts = np.random.uniform(0, 1, (N, 2))
pts = np.sort(pts, axis=1)
x = pts[:, 0]
y = pts[:, 1]

a = x
b = y - x
c = 1 - y

is_triangle = (a + b > c) & (a + c > b) & (b + c > a)
is_acute = (a**2 + b**2 > c**2) & (a**2 + c**2 > b**2) & (b**2 + c**2 > a**2)
ev_A = is_triangle & is_acute

p_sim = np.mean(ev_A)
print(f"Оценка Монте-Карло при N={N}: p_hat = {p_sim:.7f}")

freq = np.cumsum(ev_A) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq, label=f"Частота p_hat (оценка = {p_sim:.7f})", color="indigo")
ax.axhline(p_theor, color="red", linestyle="--", label=f"Теория = {p_theor:.7f}")
ax.set_title("Сходимость частоты образования остроугольного треугольника (N = 150 000)")
ax.set_xlabel("N")
ax.set_ylabel("p_hat")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.10. Случайный отрезок в круге

**Условие:**
В круге радиуса 1 наудачу выбирается отрезок. Найти вероятность того, что длина этого отрезка будет меньше радиуса круга.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_theor = 1 - 3 * np.sqrt(3) / (4 * np.pi)
print(f"Теоретическая вероятность = {p_theor:.16f}")

N = 200000
r1 = np.sqrt(np.random.uniform(0, 1, N))
th1 = np.random.uniform(0, 2 * np.pi, N)
x1, y1 = r1 * np.cos(th1), r1 * np.sin(th1)

r2 = np.sqrt(np.random.uniform(0, 1, N))
th2 = np.random.uniform(0, 2 * np.pi, N)
x2, y2 = r2 * np.cos(th2), r2 * np.sin(th2)

d = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
ev = (d < 1.0)
p_sim = np.mean(ev)
print(f"Оценка Монте-Карло при N={N}: p_hat = {p_sim:.6f}")

freq = np.cumsum(ev) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq, label=f"Относительная частота (оценка = {p_sim:.6f})", color="darkred")
ax.axhline(p_theor, color="black", linestyle="--", label=f"Теория = {p_theor:.6f}")
ax.set_title("Сходимость частоты длины случайного отрезка < 1")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.11. Встреча $n$ лиц с одинаковым временем ожидания

**Условие:**
$n$ лиц условились о встрече между 10 и 11 часами утра, причем договорились ждать друг друга не более $60\cdot q$ минут ($q < 1$). Считая, что момент прихода на встречу выбирается каждым «наудачу» в пределах указанного часа, докажите, что вероятность того, что встреча состоится, определяется равенством $P_n = n q^{n-1} - (n-1) q^n$. Проверить формулу для $n = 2$ и $n = 3$. Для $n = 3$ найти также вероятность того, что встреча по крайней мере двух лиц состоится.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

q = 0.25
N = 150000

p2_theor = 2 * q - q**2
p3_theor = 3 * q**2 - 2 * q**3
p3_at_least_2_theor = 1 - (1 - 2 * q)**3

print(f"Теория n=2: {p2_theor:.6f}")
print(f"Теория n=3 (все): {p3_theor:.6f}")
print(f"Теория n=3 (хотя бы двое): {p3_at_least_2_theor:.6f}")

t2 = np.random.uniform(0, 1, (N, 2))
ev2 = (np.max(t2, axis=1) - np.min(t2, axis=1)) <= q

t3 = np.random.uniform(0, 1, (N, 3))
ev3_all = (np.max(t3, axis=1) - np.min(t3, axis=1)) <= q

d12 = np.abs(t3[:, 0] - t3[:, 1]) <= q
d13 = np.abs(t3[:, 0] - t3[:, 2]) <= q
d23 = np.abs(t3[:, 1] - t3[:, 2]) <= q
ev3_at_least_2 = d12 | d13 | d23

print(f"Оценка n=2: {np.mean(ev2):.6f}")
print(f"Оценка n=3 (все): {np.mean(ev3_all):.6f}")
print(f"Оценка n=3 (хотя бы двое): {np.mean(ev3_at_least_2):.6f}")

freq2 = np.cumsum(ev2) / np.arange(1, N + 1)
freq3 = np.cumsum(ev3_all) / np.arange(1, N + 1)
freq3_2 = np.cumsum(ev3_at_least_2) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq2, label=f"n=2: частота (оценка = {np.mean(ev2):.4f})", color="blue")
ax.axhline(p2_theor, color="blue", linestyle="--")
ax.plot(freq3_2, label=f"n=3 (>=2): частота (оценка = {np.mean(ev3_at_least_2):.4f})", color="green")
ax.axhline(p3_at_least_2_theor, color="green", linestyle="--")
ax.plot(freq3, label=f"n=3 (все): частота (оценка = {np.mean(ev3_all):.4f})", color="purple")
ax.axhline(p3_theor, color="purple", linestyle="--")
ax.set_title("Сходимость частот встречи для n=2 и n=3 (q = 0.25)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.12. Встреча лиц с различными временами ожидания

**Условие:**
Пусть $n$ лиц имеют время ожидания $w_1, w_2, \dots, w_n$, выраженные в долях от общего интервала времени. Считая, что момент прихода на встречу выбирается каждым «наудачу» в пределах указанного часа, докажите, что вероятность того, что встреча состоится, определяется следующими равенствами для:
- a) $n = 2$: $P_2 = 1 - \frac{(1-w_1)^2}{2} - \frac{(1-w_2)^2}{2}$;
- b) $n = 3$: $P_3 = w_1 w_2 + w_1 w_3 + w_2 w_3 - \frac{1}{2}(w_1 w_2^2 + w_1 w_3^2 + w_2 w_3^2) - \frac{1}{3} w_1^3 - \frac{1}{6} w_2^3$;
- c) $n = 3, w_1 = \frac{1}{4}, w_2 = \frac{1}{3}, w_3 = \frac{1}{2}$ (15 минут, 20 минут, 30 минут): $P_3 = \frac{1435}{5184} \approx 0{,}27681$;
- d) $n = 4$: общая формула через полиномы от $w_i$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

w1, w2, w3 = 1 / 4, 1 / 3, 1 / 2
p3_theor = (
    w1 * w2 + w1 * w3 + w2 * w3
    - 0.5 * (w1 * w2**2 + w1 * w3**2 + w2 * w3**2)
    - (1 / 3) * w1**3
    - (1 / 6) * w2**3
)
print(f"Теоретическая вероятность c) P_3 = {p3_theor:.6f} (1435/5184 ≈ 0.27681)")

N = 200000
t1 = np.random.uniform(0, 1, N)
t2 = np.random.uniform(0, 1, N)
t3 = np.random.uniform(0, 1, N)

start_overlap = np.maximum(np.maximum(t1, t2), t3)
end_overlap = np.minimum(np.minimum(t1 + w1, t2 + w2), t3 + w3)
ev = start_overlap <= end_overlap

p_sim = np.mean(ev)
print(f"Оценка Монте-Карло при N={N}: p_hat = {p_sim:.6f}")

freq = np.cumsum(ev) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq, label=f"Частота (оценка = {p_sim:.5f})", color="coral")
ax.axhline(p3_theor, color="black", linestyle="--", label=f"Теория = {p3_theor:.5f}")
ax.set_title("Сходимость частоты одновременной встречи трех лиц (w1=1/4, w2=1/3, w3=1/2)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.13. Добавление белого шара в корзину с биномиальным распределением

**Условие:**
В корзину с $n$ шарами опускают белый шар. Найдите вероятность того, что извлеченный шар окажется белым, если первоначальное распределение белых шаров в корзине имеет биномиальное распределение c параметром $p$. Решите задачу для общего случая и для конкретных $n = 10, p = \frac{1}{3}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 10
p = 1 / 3
p_theor = (n * p + 1) / (n + 1)
print(f"Общая аналитическая формула: P(W) = (n*p + 1) / (n + 1)")
print(f"При n={n}, p={p:.4f}: P(W) = {p_theor:.6f} ({int(round(n*p+1)*3)}/33 = 13/33)")

N = 150000
W = np.random.binomial(n, p, size=N)
draw_prob = (W + 1) / (n + 1)
drawn_white = np.random.rand(N) < draw_prob

p_sim = np.mean(drawn_white)
print(f"Оценка Монте-Карло = {p_sim:.6f}")

freq = np.cumsum(drawn_white) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq, label=f"Частота (оценка = {p_sim:.5f})", color="seagreen")
ax.axhline(p_theor, color="red", linestyle="--", label=f"Теория = {p_theor:.5f}")
ax.set_title("Сходимость частоты извлечения белого шара (n=10, p=1/3)")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.14. Перекладывание шаров из двух корзин в третью

**Условие:**
Имеется две корзины с белыми и черными шарами. В первой корзине всего 6 шаров, при этом количество белых шаров распределено по биномиальному закону с параметрами $n = 3$ и $p = 0{,}8$. Во второй корзине имеется всего 11 шаров, при этом количество белых шаров распределено по биномиальному закону с параметрами $n = 4$ и $p = 0{,}7$. Из обеих корзин все шары перекладываются в третью корзину.
1) Какова вероятность $P(A)$, что случайно вынутый из третьей корзины шар окажется белым (событие $A$)?
2) Найдите условную вероятность $P(H|A)$, того, что случайно вынутый из третьей корзины шар первоначально находился в первой корзине (событие $H$), при условии, что он белый (событие $A$)?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

total_1 = 6
total_2 = 11
total_3 = total_1 + total_2

ew1 = 3 * 0.8
ew2 = 4 * 0.7

p_A_theor = (ew1 + ew2) / total_3
p_H_given_A_theor = ew1 / (ew1 + ew2)

print(f"Теория P(A) = {p_A_theor:.6f} (26/85 ≈ 0.3059)")
print(f"Теория P(H|A) = {p_H_given_A_theor:.6f} (6/13 ≈ 0.4615)")

N = 150000
W1 = np.random.binomial(3, 0.8, size=N)
W2 = np.random.binomial(4, 0.7, size=N)

drawn_ball_idx = np.random.randint(0, total_3, size=N)
from_basket_1 = drawn_ball_idx < total_1

is_white = np.where(from_basket_1, drawn_ball_idx < W1, (drawn_ball_idx - total_1) < W2)

p_A_sim = np.mean(is_white)
p_H_given_A_sim = np.mean(from_basket_1[is_white])

print(f"Оценка Монте-Карло P(A) = {p_A_sim:.4f}")
print(f"Оценка Монте-Карло P(H|A) = {p_H_given_A_sim:.4f}")

freq_A = np.cumsum(is_white) / np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freq_A, label=f"Частота P(A) (оценка = {p_A_sim:.4f})", color="teal")
ax.axhline(p_A_theor, color="red", linestyle="--", label=f"Теория = {p_A_theor:.4f}")
ax.set_title("Сходимость частоты извлечения белого шара из третьей корзины")
ax.set_xlabel("N")
ax.set_ylabel("Относительная частота")
ax.grid(True)
ax.legend()
plt.show()


## Задание 4.15. Две случайные точки в прямоугольной области

**Условие:**
В прямоугольной области, заданной ограничениями $|x| \le 20$ и $|y| \le 8$, случайным образом выбираются две точки: $(x_1,y_1)$ и $(x_2,y_2)$. Пусть $A$ и $B$ – события, состоящие в том, что:
- $A$ – расстояние между выбранными точками меньше 11;
- $B$ – модуль разности $|x_1 - x_2|$ меньше 14.

Найдите приближенно, методом Монте-Карло:
1) вероятность $P(A)$;
2) условную вероятность $P(A|B)$.

*Указание:* получите в заданной прямоугольной области 100000 пар точек и, используя все эти точки, найдите ответы, округляя их до одного знака после запятой.


In [ ]:
import numpy as np

N = 100000
x1 = np.random.uniform(-20, 20, N)
y1 = np.random.uniform(-8, 8, N)
x2 = np.random.uniform(-20, 20, N)
y2 = np.random.uniform(-8, 8, N)

dist = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
ev_A = dist < 11
ev_B = np.abs(x1 - x2) < 14

p_A = np.mean(ev_A)
p_A_given_B = np.mean(ev_A[ev_B])

print(f"P(A) = {p_A:.4f} (округленно: {round(p_A, 2)})")
print(f"P(A|B) = {p_A_given_B:.4f} (округленно: {round(p_A_given_B, 2)})")


## Задание 4.16. Две случайные точки внутри эллипса

**Условие:**
В области, ограниченной эллипсом $\left(\frac{u}{23}\right)^2 + \left(\frac{v}{6}\right)^2 = 1$, случайным образом выбираются две точки. Пусть $A$ и $B$ – события, состоящие в том, что:
- $A$ – расстояние между выбранными точками меньше 9,2;
- $B$ – координаты первой точки больше 0, а координаты второй точки меньше 0.

Найдите приближенно, методом Монте-Карло:
1) вероятность $P(A)$;
2) условную вероятность $P(A|B)$.

*Указание:* получите внутри заданного эллипса 100000 пар точек и, используя все эти пары точек, найдите ответы, округляя их до одного знака после запятой.


In [ ]:
import numpy as np

N = 100000
r1 = np.sqrt(np.random.uniform(0, 1, N))
th1 = np.random.uniform(0, 2 * np.pi, N)
u1, v1 = 23 * r1 * np.cos(th1), 6 * r1 * np.sin(th1)

r2 = np.sqrt(np.random.uniform(0, 1, N))
th2 = np.random.uniform(0, 2 * np.pi, N)
u2, v2 = 23 * r2 * np.cos(th2), 6 * r2 * np.sin(th2)

dist = np.sqrt((u1 - u2)**2 + (v1 - v2)**2)
ev_A = dist < 9.2
ev_B = (u1 > 0) & (v1 > 0) & (u2 < 0) & (v2 < 0)

p_A = np.mean(ev_A)
p_A_given_B = np.mean(ev_A[ev_B])

print(f"P(A) = {p_A:.4f} (округленно: {round(p_A, 2)})")
print(f"P(A|B) = {p_A_given_B:.4f} (округленно: {round(p_A_given_B, 2)})")


## Задание 4.17. Случайные треугольники в кубе

**Условие:**
В кубе объема 1 случайным образом выбираются точки $A, B$ и $C$. Пусть $R, S$ и $T$ – события, состоящие в том, что:
- $R$ – наименьший угол в треугольнике $ABC$ меньше $26{,}7^\circ$;
- $S$ – наибольший угол в треугольнике $ABC$ меньше $81{,}9^\circ$;
- $T$ – треугольник $ABC$ остроугольный.

Найдите приближенно, методом Монте-Карло:
1) условную вероятность $P(R|T)$;
2) условную вероятность $P(S|T)$.

*Указание:* получите 100000 остроугольных треугольников $ABC$ и, используя все эти треугольники, найдите ответы, округляя их до одного знака после запятой.


In [ ]:
import numpy as np

N_target = 100000
acute_list = []
batch_size = 300000

while sum(len(x) for x in acute_list) < N_target:
    A = np.random.uniform(0, 1, (batch_size, 3))
    B = np.random.uniform(0, 1, (batch_size, 3))
    C = np.random.uniform(0, 1, (batch_size, 3))
    
    a2 = np.sum((B - C)**2, axis=1)
    b2 = np.sum((A - C)**2, axis=1)
    c2 = np.sum((A - B)**2, axis=1)
    
    is_acute = (b2 + c2 > a2) & (a2 + c2 > b2) & (a2 + b2 > c2)
    
    a = np.sqrt(a2[is_acute])
    b = np.sqrt(b2[is_acute])
    c = np.sqrt(c2[is_acute])
    
    cos_A = np.clip((b**2 + c**2 - a**2) / (2 * b * c), -1.0, 1.0)
    cos_B = np.clip((a**2 + c**2 - b**2) / (2 * a * c), -1.0, 1.0)
    cos_C = np.clip((a**2 + b**2 - c**2) / (2 * a * b), -1.0, 1.0)
    
    deg_A = np.degrees(np.arccos(cos_A))
    deg_B = np.degrees(np.arccos(cos_B))
    deg_C = np.degrees(np.arccos(cos_C))
    
    angles = np.vstack([deg_A, deg_B, deg_C]).T
    acute_list.append(angles)

all_angles = np.vstack(acute_list)[:N_target]
min_angles = np.min(all_angles, axis=1)
max_angles = np.max(all_angles, axis=1)

p_R_given_T = np.mean(min_angles < 26.7)
p_S_given_T = np.mean(max_angles < 81.9)

print(f"P(R|T) = {p_R_given_T:.4f} (округленно: {round(p_R_given_T, 2)})")
print(f"P(S|T) = {p_S_given_T:.4f} (округленно: {round(p_S_given_T, 2)})")
